### Config and setup

#### Function refresh

In [ ]:
%load_ext autoreload
%autoreload 2
# %reload_ext autoreload

In [2]:
import sys
sys.path.insert(0, '../functions')

#### Library imports

In [17]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import json
import random
from pathlib import Path
from collections import Counter

import time

import seaborn as sns
import matplotlib.pyplot as plt

from captum.attr import IntegratedGradients

torch.manual_seed(25)
np.random.seed(25)
random.seed(25)

#### Functions import

In [88]:
from models import DANN
from data_loading import prepare_dataloaders, load_subject_data_flexible
from metrics import evaluate

from explainability import (
    DANNClassifierWrapper, load_model_for_xai,
    fold_batchnorm, verify_folding,
    lrp_epsilon_conv2d, lrp_zplus_conv2d,
    lrp_linear, lrp_avgpool,
    compute_lrp, run_lrp_full,
    load_lrp_attributions,
    compute_per_class_accuracy_106,
    compute_deeplift, run_deeplift_full,
    load_deeplift_attributions,
    compute_shap, run_shap_full,
    load_shap_attributions,
    compute_ig, run_ig_full,
    load_ig_attributions,
    run_and_save_misclassified_lrp,
    run_misclassified_lrp_single
    )

from xai_utils import (
    assign_score_groups, get_grouped_order,
    get_score_grouped_order, classify_failure_groups,
    compute_lrp_deeplift_agreement,
    compute_method_agreement,
    compute_within_group_consistency,
    print_consistency_results,
    ch16_list
)

from xai_plots import (
    plot_group_mean_attribution,
    plot_stopping_criterion_consistency
)

#### Path

In [21]:
# common path
base_path = Path("..")
processed_path = base_path / "Data/processed"
results_path = base_path / "results"
ch16_path = base_path / "Data/processed_16ch"

# existing attribution folders for loading LRP and DeepLIFT
lrp_attr_path = results_path / "lrp_attributions"
binary_attr_path = lrp_attr_path / "binary_cdan_16ch"
multiclass_attr_path = lrp_attr_path / "multiclass_16ch"

# deep lift path
deeplift_attr_path = results_path / "deeplift_attributions"
binary_deeplift_path = deeplift_attr_path / "binary_cdan_16ch"
multiclass_deeplift_path = deeplift_attr_path / "multiclass_16ch"

# per class path
per_class_path = results_path / "per_class_accuracy"

# shap folders
shap_attr_path = results_path / "shap_attributions"
binary_shap_path = shap_attr_path / "binary_cdan_16ch"
multiclass_shap_path = shap_attr_path / "multiclass_16ch"

# IG folders
ig_attr_path = results_path / "ig_attributions"
binary_ig_zero_path = ig_attr_path / "binary_cdan_16ch_zero"
binary_ig_mean_path = ig_attr_path / "binary_cdan_16ch_mean"
multiclass_ig_zero_path = ig_attr_path / "multiclass_16ch_zero"
multiclass_ig_mean_path = ig_attr_path / "multiclass_16ch_mean"

# only creating zero baseline folders since mean is dropped
binary_ig_zero_path.mkdir(parents=True, exist_ok=True)
multiclass_ig_zero_path.mkdir(parents=True, exist_ok=True)

In [6]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

#### 16 channel importing from xai_utils

In [7]:
# ch16_list = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4',
#              'CP5', 'CP6', 'P3', 'P4', 'C5', 'C6', 'P5', 'P6']

#### Valid Subjects

In [8]:
# 106 subjects after removing S088, S092, S100 (incompatible sampling rate)
excluded = ['S088', 'S092', 'S100']

valid_subjects = []
for i in range(1, 110):
    subject_id = f'S{str(i).zfill(3)}'
    subject_file = processed_path / f'{subject_id}_processed.npz'
    if subject_file.exists() and subject_id not in excluded:
        valid_subjects.append(subject_id)

print(f"Valid subjects: {len(valid_subjects)}")
print(f"First five: {valid_subjects[:5]}")
print(f"Last five: {valid_subjects[-5:]}")

Valid subjects: 106
First five: ['S001', 'S002', 'S003', 'S004', 'S005']
Last five: ['S105', 'S106', 'S107', 'S108', 'S109']


#### Setup cell for groupings rebuild

In [9]:
consistency_subjects = ['S007', 'S085', 'S108', 'S107', 'S080',
                         'S097', 'S037', 'S006', 'S039', 'S018']

In [10]:
# load kappas
with open(results_path / 'loso_results_cdan_16ch_RETRAIN.json') as f:
    cdan_results = json.load(f)

all_kappas = {
    subj: cdan_results[subj]['kappa_stopping']['final_kappa']
    for subj in valid_subjects
}

# load per-class accuracy
with open(per_class_path / 'multiclass_16ch_per_class_accuracy.json') as f:
    per_class_data = json.load(f)

all_per_class = {}
for subj in valid_subjects:
    pca = per_class_data[subj]['kappa']['per_class_accuracy']
    all_per_class[subj] = {int(cls): float(acc) for cls, acc in pca.items()}

# rebuild groupings
score_groups, boundaries = assign_score_groups(all_kappas, std_multiplier=0.7)

group_a, group_b, neither = classify_failure_groups(all_per_class, all_kappas)
group_a_subjects = [s['subject'] for s in group_a]
group_b_subjects = [s['subject'] for s in group_b]

category_groups_106 = {}
for subj in valid_subjects:
    if all_kappas[subj] < 0.1:
        category_groups_106[subj] = 'negative_transfer'
    elif subj in group_a_subjects:
        category_groups_106[subj] = 'group_a_collapse'
    elif subj in group_b_subjects:
        category_groups_106[subj] = 'group_b_flat'
    else:
        category_groups_106[subj] = 'neither'

xai_subjects = {
    "top": ["S007", "S085"],
    "bottom_kappa": ["S018", "S097", "S107"],
    "group_a_collapse": ["S037", "S006"],
    "group_b_flat": ["S039", "S020"],
    "typical": ["S080", "S065", "S030", "S108"],
}
xai_subset = [s for group in xai_subjects.values() for s in group]

# load existing attributions for comparison
binary_attrs_106 = load_lrp_attributions(
    binary_attr_path, valid_subjects, n_classes=2)

mc_attrs_106 = load_lrp_attributions(
    multiclass_attr_path, valid_subjects, n_classes=4)

binary_dl_106 = load_deeplift_attributions(
    binary_deeplift_path, valid_subjects, n_classes=2)

mc_dl_106 = load_deeplift_attributions(
    multiclass_deeplift_path, valid_subjects, n_classes=4)

binary_shap_106 = load_shap_attributions(
    binary_shap_path, valid_subjects, n_classes=2)

mc_shap_106 = load_shap_attributions(
    multiclass_shap_path, valid_subjects, n_classes=4)

binary_ig_106 = load_ig_attributions(
    binary_ig_zero_path, valid_subjects, n_classes=2)

mc_ig_106 = load_ig_attributions(
    multiclass_ig_zero_path, valid_subjects, n_classes=4)

print("Setup complete")
print(f"Score groups: {dict(Counter(score_groups.values()))}")
print(f"Category groups: {dict(Counter(category_groups_106.values()))}")
print(f"XAI subset: {len(xai_subset)} subjects")
print(f"LRP binary: {len(binary_attrs_106)}, DeepLIFT: {len(binary_dl_106)}, "
      f"SHAP: {len(binary_shap_106)}, IG: {len(binary_ig_106)}")

Loaded: 106/106 subjects
Loaded: 106/106 subjects
Loaded: 106/106 subjects
Loaded: 106/106 subjects
Loaded: 106/106 subjects
Loaded: 106/106 subjects
Loaded: 106/106 subjects
Loaded: 106/106 subjects
Setup complete
Score groups: {'mid_high': 21, 'high': 30, 'negative_transfer': 3, 'low': 28, 'mid_low': 24}
Category groups: {'neither': 87, 'negative_transfer': 3, 'group_a_collapse': 13, 'group_b_flat': 3}
XAI subset: 13 subjects
LRP binary: 106, DeepLIFT: 106, SHAP: 106, IG: 106


### End of config and setup

#### Functions

##### build_binary_relevance_table

In [ ]:
def build_binary_relevance_table(attrs_106, score_groups, ch16_names,
                                  method_name=""):
    """
    Build mean relevance score table for binary task, score grouping.
    Returns a long-format DataFrame with columns:
    [group, class, channel, mean_relevance]
    """
    channel_avg = {}
    for subj in attrs_106:
        channel_avg[subj] = {
            'class_0': np.mean(
                attrs_106[subj]['kappa']['mean_by_class'][0], axis=1),
            'class_1': np.mean(
                attrs_106[subj]['kappa']['mean_by_class'][1], axis=1)
        }

    rows = []
    for group in ['high', 'mid_high', 'mid_low', 'low', 'negative_transfer']:
        group_subjects = [s for s, g in score_groups.items() if g == group]
        for cls, cls_name in [(0, 'left'), (1, 'right')]:
            group_attrs = np.array([channel_avg[s][f'class_{cls}']
                                    for s in group_subjects])
            group_mean = np.mean(group_attrs, axis=0)
            for ch_idx, ch_name in enumerate(ch16_names):
                rows.append({
                    'group': group,
                    'class': cls_name,
                    'channel': ch_name,
                    'mean_relevance': round(group_mean[ch_idx], 6)
                })

    return pd.DataFrame(rows)

##### build_multiclass_relevance_table

In [ ]:
def build_multiclass_relevance_table(attrs_106, category_groups, ch16_names,
                                      method_name=""):
    """
    Build mean relevance score table for multiclass task, category grouping.
    Handles empty arrays (e.g. S018 class 2) by replacing with zeros.
    Returns a long-format DataFrame with columns:
    [group, class, channel, mean_relevance]
    """
    class_names = {0: 'left', 1: 'right', 2: 'both_fists', 3: 'both_feet'}
    n_classes = 4

    channel_avg = {}
    for subj in attrs_106:
        channel_avg[subj] = {}
        for cls in range(n_classes):
            arr = attrs_106[subj]['kappa']['mean_by_class'][cls]
            if np.array(arr).ndim == 2:
                channel_avg[subj][cls] = np.mean(arr, axis=1)
            else:
                channel_avg[subj][cls] = np.zeros(len(ch16_names))

    rows = []
    for group in ['neither', 'group_a_collapse',
                  'group_b_flat', 'negative_transfer']:
        group_subjects = [s for s, g in category_groups.items() if g == group]
        for cls in range(n_classes):
            group_attrs = np.array([channel_avg[s][cls]
                                    for s in group_subjects])
            group_mean = np.mean(group_attrs, axis=0)
            for ch_idx, ch_name in enumerate(ch16_names):
                rows.append({
                    'group': group,
                    'class': class_names[cls],
                    'channel': ch_name,
                    'mean_relevance': round(group_mean[ch_idx], 6)
                })

    return pd.DataFrame(rows)

##### get_top_bottom_channels

In [ ]:
def get_top_bottom_channels(relevance_df, class_names, n=3,
                             group_order=None):
    """
    For each group and class, print top n and bottom n channels
    by mean relevance score.
    """
    if group_order is None:
        group_order = relevance_df['group'].unique().tolist()

    for cls_name in class_names:
        print(f"\nClass: {cls_name}")
        print("-" * 60)
        for group in group_order:
            subset = relevance_df[
                (relevance_df['class'] == cls_name) &
                (relevance_df['group'] == group)
            ].sort_values('mean_relevance', ascending=False)

            if subset.empty:
                continue

            top_n = subset.head(n)[['channel', 'mean_relevance']].values
            bot_n = subset.tail(n)[['channel', 'mean_relevance']].values

            print(f"\n  {group}:")
            print(f"    Top {n}:    ", end="")
            for ch, val in top_n:
                print(f"{ch}={val:.6f}  ", end="")
            print(f"\n    Bottom {n}: ", end="")
            for ch, val in bot_n:
                print(f"{ch}={val:.6f}  ", end="")
        print()

### Channel Checks with npz files 


In [14]:
import numpy as np
from pathlib import Path

# ch16_path = Path("../Data/processed_16ch")

# Check keys in one subject file
data = np.load(ch16_path / "S001_16ch.npz")
print("Keys in npz file:", list(data.keys()))

Keys in npz file: ['data', 'labels']


In [ ]:
# Load one subject
data = np.load(ch16_path / "S001_16ch.npz")
X = data['data']
y = data['labels']

print("Data shape:", X.shape)
print("Labels shape:", y.shape)
print("Expected: (n_trials, 16, 480)\n")

# Channel names in the exact order saved during preprocessing
ch16_names = [
    'C3', 'Cz', 'C4',
    'FC3', 'FCz',
    'CP3', 'CPz', 'CP4',
    'CP5', 'CP6',
    'P3', 'P4',
    'C5', 'C6',
    'P5', 'P6'
]

print("Channel index to name mapping:")
for idx, name in enumerate(ch16_names):
    left = ['C3', 'FC3', 'CP3', 'CP5', 'P3', 'C5', 'P5']
    right = ['C4', 'CP4', 'CP6', 'P4', 'C6', 'P6']
    midline = ['Cz', 'FCz', 'CPz']
    hemisphere = "LEFT" if name in left else \
                 "RIGHT" if name in right else "MIDLINE"
    print(f"  Index {idx:2d} → {name:6s} ({hemisphere})")

Data shape: (44, 16, 480)
Labels shape: (44,)
Expected: (n_trials, 16, 480)

Channel index to name mapping:


#### Channel confirmation in dataframe

In [18]:
channel_df = pd.DataFrame({
    'Index': range(16),
    'Channel': ch16_names,
    'Hemisphere': [
        'LEFT' if name in ['C3', 'FC3', 'CP3', 'CP5', 'P3', 'C5', 'P5']
        else 'RIGHT' if name in ['C4', 'CP4', 'CP6', 'P4', 'C6', 'P6']
        else 'MIDLINE'
        for name in ch16_names
    ]
})

print(channel_df.to_string(index=False))

 Index Channel Hemisphere
     0      C3       LEFT
     1      Cz    MIDLINE
     2      C4      RIGHT
     3     FC3       LEFT
     4     FCz    MIDLINE
     5     CP3       LEFT
     6     CPz    MIDLINE
     7     CP4      RIGHT
     8     CP5       LEFT
     9     CP6      RIGHT
    10      P3       LEFT
    11      P4      RIGHT
    12      C5       LEFT
    13      C6      RIGHT
    14      P5       LEFT
    15      P6      RIGHT


### Relevance score table

#### binary

In [44]:
# base result file - all analysis and plots use this file
with open(results_path / 'loso_results_cdan_16ch_RETRAIN.json') as f:
    binary_results = json.load(f)

first_key = list(binary_results.keys())[0]
first_value = binary_results[first_key]

print("First key:", first_key)
print("\nTop level keys:", list(first_value.keys()))
print("kappa_stopping keys:", list(first_value['kappa_stopping'].keys()))

First key: S001

Top level keys: ['kappa_stopping', 'composite_stopping']
kappa_stopping keys: ['final_accuracy', 'final_kappa', 'final_auc', 'log']


In [45]:
# Extract binary kappa and accuracy for all subjects
binary_kappa = {}
binary_accuracy = {}
binary_composite_kappa = {}

for subj in valid_subjects:
    if subj not in binary_results:
        continue
    binary_kappa[subj] = binary_results[subj]['kappa_stopping']['final_kappa']
    binary_accuracy[subj] = binary_results[subj]['kappa_stopping']['final_accuracy']
    binary_composite_kappa[subj] = binary_results[subj]['composite_stopping']['final_kappa']

print(f"Subjects loaded: {len(binary_kappa)}")
print(f"Kappa range: {min(binary_kappa.values()):.4f} to {max(binary_kappa.values()):.4f}")

# Negative transfer: kappa < 0.1
negative_transfer = [s for s, k in binary_kappa.items() if k < 0.1]
print(f"\nNegative transfer (kappa < 0.1): {len(negative_transfer)} subjects")
print(negative_transfer)

# Negative transfer: composite < 0.1
negative_transfer = [s for s, k in binary_composite_kappa.items() if k < 0.1]
print(f"Negative transfer (composite kappa < 0.1): {len(negative_transfer)} subjects")
print(sorted(negative_transfer))

Subjects loaded: 106
Kappa range: -0.0177 to 1.0000

Negative transfer (kappa < 0.1): 3 subjects
['S003', 'S005', 'S018']
Negative transfer (composite kappa < 0.1): 2 subjects
['S003', 'S018']


#### LRP - Score grouping

In [57]:
ch16_list = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4',
             'CP5', 'CP6', 'P3', 'P4', 'C5', 'C6', 'P5', 'P6']

In [58]:
# Build per-subject channel averages
subj_channel_avg = {}
for subj in valid_subjects:
    subj_channel_avg[subj] = {
        'class_0': np.mean(binary_attrs_106[subj]['kappa']['mean_by_class'][0], axis=1),
        'class_1': np.mean(binary_attrs_106[subj]['kappa']['mean_by_class'][1], axis=1)
    }

# Build table: mean relevance per channel per score group
rows = []
for group in ['high', 'mid_high', 'mid_low', 'low', 'negative_transfer']:
    group_subjects = [s for s, g in score_groups.items() if g == group]
    for cls, cls_name in [(0, 'left'), (1, 'right')]:
        group_attrs = np.array([subj_channel_avg[s][f'class_{cls}']
                                for s in group_subjects])
        group_mean = np.mean(group_attrs, axis=0)
        for ch_idx, ch_name in enumerate(ch16_names):
            rows.append({
                'group': group,
                'class': cls_name,
                'channel': ch_name,
                'mean_relevance': round(group_mean[ch_idx], 6)
            })

relevance_df = pd.DataFrame(rows)

# Pivot to readable format: channels as rows, groups as columns
for cls_name in ['left', 'right']:
    print(f"\nClass: {cls_name}")
    pivot = relevance_df[relevance_df['class'] == cls_name].pivot(
        index='channel', columns='group', values='mean_relevance'
    )[['high', 'mid_high', 'mid_low', 'low', 'negative_transfer']]
    pivot = pivot.reindex(ch16_names)
    print(pivot.to_string())


Class: left
group        high  mid_high   mid_low       low  negative_transfer
channel                                                           
C3       0.001934  0.001806  0.001634  0.001475           0.000926
Cz      -0.000001 -0.000024 -0.000039  0.000114          -0.000125
C4      -0.001266 -0.001246 -0.001187 -0.001267          -0.000924
FC3      0.000251  0.000242  0.000170  0.000246           0.000308
FCz     -0.000036 -0.000059 -0.000039 -0.000009          -0.000106
CP3      0.002259  0.001903  0.001831  0.001578           0.001156
CPz     -0.000004 -0.000022  0.000062  0.000048           0.000192
CP4     -0.001225 -0.001246 -0.001176 -0.001348          -0.000868
CP5      0.000190  0.000210  0.000129  0.000233           0.000216
CP6     -0.000309 -0.000262 -0.000217 -0.000324          -0.000220
P3       0.000502  0.000341  0.000307  0.000313           0.000181
P4      -0.000338 -0.000242 -0.000255 -0.000178          -0.000163
C5      -0.000124 -0.000133 -0.000118 -0.000018  

#### Category grouping

In [74]:
class_names = {0: 'left', 1: 'right', 2: 'both_fists', 3: 'both_feet'}

# Build per-subject channel averages for multiclass
mc_channel_avg = {}
for subj in valid_subjects:
    mc_channel_avg[subj] = {}
    for cls in range(4):
        arr = mc_attrs_106[subj]['kappa']['mean_by_class'][cls]
        if np.array(arr).ndim == 2:
            mc_channel_avg[subj][cls] = np.mean(arr, axis=1)
        else:
            mc_channel_avg[subj][cls] = np.zeros(16)
            print(f"Warning: {subj} class {cls} empty, replaced with zeros")

# Build table: mean relevance per channel per category group
rows_mc = []
for group in ['neither', 'group_a_collapse', 'group_b_flat', 'negative_transfer']:
    group_subjects = [s for s, g in category_groups_106.items() if g == group]
    for cls in range(4):
        group_attrs = np.array([mc_channel_avg[s][cls]
                                for s in group_subjects])
        group_mean = np.mean(group_attrs, axis=0)
        for ch_idx, ch_name in enumerate(ch16_names):
            rows_mc.append({
                'group': group,
                'class': class_names[cls],
                'channel': ch_name,
                'mean_relevance': round(group_mean[ch_idx], 6)
            })

relevance_df_mc = pd.DataFrame(rows_mc)

for cls_name in ['left', 'right', 'both_fists', 'both_feet']:
    print(f"\nClass: {cls_name}")
    pivot = relevance_df_mc[relevance_df_mc['class'] == cls_name].pivot(
        index='channel', columns='group', values='mean_relevance'
    )[['neither', 'group_a_collapse', 'group_b_flat', 'negative_transfer']]
    pivot = pivot.reindex(ch16_names)
    pivot.columns = ['Neither', 'Group A', 'Group B', 'Neg. Transfer']
    print(pivot.round(6).to_string())


Class: left
          Neither   Group A   Group B  Neg. Transfer
channel                                             
C3       0.002384  0.002212  0.001908       0.001804
Cz       0.000087  0.000298  0.000170       0.000280
C4      -0.001850 -0.001817 -0.002782      -0.001573
FC3      0.000041 -0.000112 -0.000081      -0.000026
FCz     -0.000006  0.000036  0.000082      -0.000034
CP3      0.002162  0.001900  0.002270       0.001245
CPz      0.000476  0.000268  0.000620       0.000450
CP4     -0.001436 -0.001425 -0.001241      -0.001297
CP5      0.000090  0.000010  0.000436       0.000110
CP6     -0.000299 -0.000238 -0.000123      -0.000351
P3       0.000419  0.000423  0.001153       0.000377
P4      -0.000174 -0.000212 -0.000053      -0.000201
C5      -0.000207 -0.000107 -0.000277      -0.000119
C6      -0.000072 -0.000152 -0.000018      -0.000039
P5      -0.000030  0.000018 -0.000419       0.000015
P6      -0.000101 -0.000218 -0.000265      -0.000313

Class: right
          Neither  

##### debuging 
category grouping and for s018 as both_fist class is empty

In [ ]:
# Category grouping - uses binary kappas + multiclass per class accuracy
# Exactly mirrors 04_xai_analysis setup cell

group_a, group_b, neither = classify_failure_groups(
    all_per_class, binary_kappa)

group_a_subjects = [s['subject'] for s in group_a]
group_b_subjects = [s['subject'] for s in group_b]

category_groups_106 = {}
for subj in valid_subjects:
    if binary_kappa[subj] < 0.1:
        category_groups_106[subj] = 'negative_transfer'
    elif subj in group_a_subjects:
        category_groups_106[subj] = 'group_a_collapse'
    elif subj in group_b_subjects:
        category_groups_106[subj] = 'group_b_flat'
    else:
        category_groups_106[subj] = 'neither'

from collections import Counter
print("Category group distribution:")
print(Counter(category_groups_106.values()))

Category group distribution:
Counter({'neither': 87, 'group_a_collapse': 13, 'negative_transfer': 3, 'group_b_flat': 3})


In [ ]:
mc_attrs_106 = load_lrp_attributions(
    multiclass_attr_path, valid_subjects, n_classes=4)

print(f"Multiclass LRP loaded: {len(mc_attrs_106)} subjects")
sample_mc = mc_attrs_106['S001']
print("Top level keys:", list(sample_mc.keys()))
print("Kappa keys:", list(sample_mc['kappa'].keys()))
print("mean_by_class keys:", list(sample_mc['kappa']['mean_by_class'].keys()))
first_class_key = list(sample_mc['kappa']['mean_by_class'].keys())[0]
print(f"Shape of class {first_class_key}:", sample_mc['kappa']['mean_by_class'][first_class_key].shape)

Loaded: 106/106 subjects
Multiclass LRP loaded: 106 subjects
Top level keys: ['kappa', 'composite']
Kappa keys: ['mean_by_class', 'n_trials']
mean_by_class keys: [0, 1, 2, 3]
Shape of class 0: (16, 480)


In [ ]:
sample_mc = mc_attrs_106['S001']['kappa']['mean_by_class']
for cls in range(4):
    print(f"Class {cls}: type={type(sample_mc[cls])}, shape={np.array(sample_mc[cls]).shape}")

Class 0: type=<class 'numpy.ndarray'>, shape=(16, 480)
Class 1: type=<class 'numpy.ndarray'>, shape=(16, 480)
Class 2: type=<class 'numpy.ndarray'>, shape=(16, 480)
Class 3: type=<class 'numpy.ndarray'>, shape=(16, 480)


In [75]:
sample_s018 = mc_attrs_106['S018']['kappa']['mean_by_class']
for cls in range(4):
    arr = np.array(sample_s018[cls])
    print(f"S018 class {cls}: shape={arr.shape}, ndim={arr.ndim}")

S018 class 0: shape=(16, 480), ndim=2
S018 class 1: shape=(16, 480), ndim=2
S018 class 2: shape=(), ndim=0
S018 class 3: shape=(16, 480), ndim=2


In [76]:
# Check what happens when we stack S018 class 2 with others
neg_transfer_subjects = [s for s, g in category_groups_106.items()
                         if g == 'negative_transfer']
print("Negative transfer subjects:", neg_transfer_subjects)

for subj in neg_transfer_subjects:
    arr = mc_attrs_106[subj]['kappa']['mean_by_class'][2]
    print(f"{subj} class 2: shape={np.array(arr).shape}")

Negative transfer subjects: ['S003', 'S005', 'S018']
S003 class 2: shape=(16, 480)
S005 class 2: shape=(16, 480)
S018 class 2: shape=()


In [71]:
# Check xai_plots.py plot_group_mean_attribution function
import inspect
from xai_plots import plot_group_mean_attribution
print(inspect.getsource(plot_group_mean_attribution))

def plot_group_mean_attribution(
    binary_attrs_106,
    category_groups_106,
    score_groups,
    valid_subjects,
    all_kappas,
    sfreq=160,
    save=True,
    title_tag="LRP binary CDAN-16ch",
):
    """
    Group-level mean attribution for all 106 subjects.
    Two figures: category groups and score groups.
    Rows = groups, columns = classes (left, right for binary).
    Each cell = mean attribution averaged across all subjects in that group.
    """
    # derive n_classes and class_names from actual data
    n_classes = len(next(iter(binary_attrs_106.values()))["kappa"]["mean_by_class"])
    class_names = (
        ["left", "right"]
        if n_classes == 2
        else ["left", "right", "both-fists", "both-feet"]
    )

    # category group order and labels
    cat_group_order = [
        "negative_transfer",
        "group_a_collapse",
        "group_b_flat",
        "neither",
    ]
    cat_group_labels = {
        "negative_transfer": f'negative transfer (n={sum(1 for

#### per score feature group analysis

In [77]:
# Check what relevance_df contains
print(relevance_df.columns.tolist())
print(relevance_df.head(3))
print(f"Groups: {relevance_df['group'].unique()}")
print(f"Classes: {relevance_df['class'].unique()}")

['group', 'class', 'channel', 'mean_relevance']
  group class channel  mean_relevance
0  high  left      C3        0.001934
1  high  left      Cz       -0.000001
2  high  left      C4       -0.001266
Groups: <StringArray>
['high', 'mid_high', 'mid_low', 'low', 'negative_transfer']
Length: 5, dtype: str
Classes: <StringArray>
['left', 'right']
Length: 2, dtype: str


In [80]:
# Run for binary
print("=" * 60)
print("BINARY - Score grouping")
print("=" * 60)
get_top_bottom_channels(
    relevance_df,
    class_names=['left', 'right'],
    group_order=['high', 'mid_high', 'mid_low', 'low', 'negative_transfer']
)

# Run for multiclass
print("\n" + "=" * 60)
print("MULTICLASS - Category grouping")
print("=" * 60)
get_top_bottom_channels(
    relevance_df_mc,
    class_names=['left', 'right', 'both_fists', 'both_feet'],
    group_order=['neither', 'group_a_collapse', 'group_b_flat',
                 'negative_transfer']
)

BINARY - Score grouping

Class: left
------------------------------------------------------------

  high:
    Top 3:    CP3=0.002259  C3=0.001934  P3=0.000502  
    Bottom 3: P4=-0.000338  CP4=-0.001225  C4=-0.001266  
  mid_high:
    Top 3:    CP3=0.001903  C3=0.001806  P3=0.000341  
    Bottom 3: CP6=-0.000262  C4=-0.001246  CP4=-0.001246  
  mid_low:
    Top 3:    CP3=0.001831  C3=0.001634  P3=0.000307  
    Bottom 3: P4=-0.000255  CP4=-0.001176  C4=-0.001187  
  low:
    Top 3:    CP3=0.001578  C3=0.001475  P3=0.000313  
    Bottom 3: CP6=-0.000324  C4=-0.001267  CP4=-0.001348  
  negative_transfer:
    Top 3:    CP3=0.001156  C3=0.000926  FC3=0.000308  
    Bottom 3: P6=-0.000309  CP4=-0.000868  C4=-0.000924  

Class: right
------------------------------------------------------------

  high:
    Top 3:    C4=0.002333  CP4=0.002109  P4=0.000359  
    Bottom 3: FC3=-0.000418  C3=-0.001261  CP3=-0.001293  
  mid_high:
    Top 3:    C4=0.002067  CP4=0.002047  CP6=0.000286  
    Bott

#### Deeplift relevance Score

##### Checks and debug

In [93]:
from explainability import (
    load_deeplift_attributions,
    load_shap_attributions,
    load_ig_attributions
)

# Quick check - load DeepLIFT binary for one subject
sample_dl = load_deeplift_attributions(binary_deeplift_path, ['S001'], n_classes=2)
print("Keys:", list(sample_dl['S001'].keys()))
print("Kappa keys:", list(sample_dl['S001']['kappa'].keys()))

Loaded: 1/1 subjects
Keys: ['kappa', 'composite']
Kappa keys: ['mean_by_class', 'n_trials']


In [94]:
print("mean_by_class keys:", list(sample_dl['S001']['kappa']['mean_by_class'].keys()))
first_key = list(sample_dl['S001']['kappa']['mean_by_class'].keys())[0]
print(f"Shape of class {first_key}:", np.array(sample_dl['S001']['kappa']['mean_by_class'][first_key]).shape)

mean_by_class keys: [0, 1]
Shape of class 0: (16, 480)


#### Deeplift - Binary

In [95]:
# Load DeepLIFT binary all 106 subjects
binary_dl_106 = load_deeplift_attributions(
    binary_deeplift_path, valid_subjects, n_classes=2)
print(f"Loaded: {len(binary_dl_106)} subjects")

# Build per-subject channel averages
dl_channel_avg = {}
for subj in valid_subjects:
    dl_channel_avg[subj] = {
        'class_0': np.mean(binary_dl_106[subj]['kappa']['mean_by_class'][0], axis=1),
        'class_1': np.mean(binary_dl_106[subj]['kappa']['mean_by_class'][1], axis=1)
    }

# Build relevance score table
rows_dl = []
for group in ['high', 'mid_high', 'mid_low', 'low', 'negative_transfer']:
    group_subjects = [s for s, g in score_groups.items() if g == group]
    for cls, cls_name in [(0, 'left'), (1, 'right')]:
        group_attrs = np.array([dl_channel_avg[s][f'class_{cls}']
                                for s in group_subjects])
        group_mean = np.mean(group_attrs, axis=0)
        for ch_idx, ch_name in enumerate(ch16_names):
            rows_dl.append({
                'group': group,
                'class': cls_name,
                'channel': ch_name,
                'mean_relevance': round(group_mean[ch_idx], 6)
            })

relevance_df_dl = pd.DataFrame(rows_dl)

# Print pivot tables
for cls_name in ['left', 'right']:
    print(f"\nDeepLIFT - Class: {cls_name}")
    pivot = relevance_df_dl[relevance_df_dl['class'] == cls_name].pivot(
        index='channel', columns='group', values='mean_relevance'
    )[['high', 'mid_high', 'mid_low', 'low', 'negative_transfer']]
    pivot = pivot.reindex(ch16_names)
    pivot.columns = ['High', 'Mid-High', 'Mid-Low', 'Low', 'Neg. Transfer']
    print(pivot.to_string())

Loaded: 106/106 subjects
Loaded: 106 subjects

DeepLIFT - Class: left
             High  Mid-High   Mid-Low       Low  Neg. Transfer
channel                                                       
C3       0.002236  0.001958  0.002005  0.001681       0.000947
Cz      -0.000004 -0.000011 -0.000045  0.000153      -0.000115
C4      -0.001431 -0.001293 -0.001454 -0.001471      -0.001003
FC3      0.000317  0.000284  0.000253  0.000330       0.000304
FCz     -0.000032 -0.000072 -0.000039  0.000024      -0.000096
CP3      0.002617  0.002050  0.002241  0.001750       0.001180
CPz      0.000019  0.000008  0.000129  0.000067       0.000194
CP4     -0.001398 -0.001332 -0.001449 -0.001571      -0.000908
CP5      0.000224  0.000240  0.000184  0.000279       0.000144
CP6     -0.000333 -0.000258 -0.000257 -0.000349      -0.000233
P3       0.000558  0.000362  0.000381  0.000350       0.000171
P4      -0.000357 -0.000252 -0.000301 -0.000196      -0.000165
C5      -0.000152 -0.000137 -0.000138 -0.000029 

#### Deeplift Multiclass 

##### Deebug and checks

In [100]:
sample_mc_dl = load_deeplift_attributions(
    multiclass_deeplift_path, ['S001'], n_classes=4)
print("Keys:", list(sample_mc_dl['S001'].keys()))
print("Kappa keys:", list(sample_mc_dl['S001']['kappa'].keys()))
print("mean_by_class keys:", list(sample_mc_dl['S001']['kappa']['mean_by_class'].keys()))
first_key = list(sample_mc_dl['S001']['kappa']['mean_by_class'].keys())[0]
print(f"Shape of class {first_key}:", np.array(sample_mc_dl['S001']['kappa']['mean_by_class'][first_key]).shape)

Loaded: 1/1 subjects
Keys: ['kappa', 'composite']
Kappa keys: ['mean_by_class', 'n_trials']
mean_by_class keys: [0, 1, 2, 3]
Shape of class 0: (16, 480)


In [101]:
sample_s018_dl = load_deeplift_attributions(
    multiclass_deeplift_path, ['S018'], n_classes=4)
for cls in range(4):
    arr = np.array(sample_s018_dl['S018']['kappa']['mean_by_class'][cls])
    print(f"S018 class {cls}: shape={arr.shape}, ndim={arr.ndim}")

Loaded: 1/1 subjects
S018 class 0: shape=(16, 480), ndim=2
S018 class 1: shape=(16, 480), ndim=2
S018 class 2: shape=(), ndim=0
S018 class 3: shape=(16, 480), ndim=2


##### Deeplift Multiclass 

In [102]:
# Load DeepLIFT multiclass all 106 subjects
mc_dl_106 = load_deeplift_attributions(
    multiclass_deeplift_path, valid_subjects, n_classes=4)
print(f"Loaded: {len(mc_dl_106)} subjects")

# Build per-subject channel averages with zeros for empty arrays
mc_dl_channel_avg = {}
for subj in valid_subjects:
    mc_dl_channel_avg[subj] = {}
    for cls in range(4):
        arr = mc_dl_106[subj]['kappa']['mean_by_class'][cls]
        if np.array(arr).ndim == 2:
            mc_dl_channel_avg[subj][cls] = np.mean(arr, axis=1)
        else:
            mc_dl_channel_avg[subj][cls] = np.zeros(16)
            print(f"Warning: {subj} class {cls} empty, replaced with zeros")

# Build table
class_names_mc = {0: 'left', 1: 'right', 2: 'both_fists', 3: 'both_feet'}
rows_mc_dl = []
for group in ['neither', 'group_a_collapse', 'group_b_flat', 'negative_transfer']:
    group_subjects = [s for s, g in category_groups_106.items() if g == group]
    for cls in range(4):
        group_attrs = np.array([mc_dl_channel_avg[s][cls]
                                for s in group_subjects])
        group_mean = np.mean(group_attrs, axis=0)
        for ch_idx, ch_name in enumerate(ch16_names):
            rows_mc_dl.append({
                'group': group,
                'class': class_names_mc[cls],
                'channel': ch_name,
                'mean_relevance': round(group_mean[ch_idx], 6)
            })

relevance_df_mc_dl = pd.DataFrame(rows_mc_dl)

for cls_name in ['left', 'right', 'both_fists', 'both_feet']:
    print(f"\nDeepLIFT Multiclass - Class: {cls_name}")
    pivot = relevance_df_mc_dl[relevance_df_mc_dl['class'] == cls_name].pivot(
        index='channel', columns='group', values='mean_relevance'
    )[['neither', 'group_a_collapse', 'group_b_flat', 'negative_transfer']]
    pivot = pivot.reindex(ch16_names)
    pivot.columns = ['Neither', 'Group A', 'Group B', 'Neg. Transfer']
    print(pivot.to_string())

Loaded: 106/106 subjects
Loaded: 106 subjects

DeepLIFT Multiclass - Class: left
          Neither   Group A   Group B  Neg. Transfer
channel                                             
C3       0.003146  0.002656  0.002766       0.002606
Cz       0.000054  0.000293  0.000026       0.000264
C4      -0.001861 -0.001850 -0.002573      -0.001792
FC3      0.000010 -0.000151 -0.000164      -0.000034
FCz      0.000007  0.000083  0.000044      -0.000015
CP3      0.002788  0.002230  0.003219       0.001796
CPz      0.000165  0.000063  0.000050       0.000136
CP4     -0.001468 -0.001427 -0.001330      -0.001509
CP5      0.000138  0.000059  0.000503       0.000159
CP6     -0.000253 -0.000195 -0.000138      -0.000367
P3       0.000433  0.000427  0.001035       0.000398
P4      -0.000263 -0.000295 -0.000203      -0.000317
C5      -0.000306 -0.000153 -0.000486      -0.000212
C6      -0.000099 -0.000159 -0.000177      -0.000078
P5      -0.000065  0.000015 -0.000290      -0.000042
P6      -0.000194 

#### SHAP Relevance score

##### Debug and checks

In [96]:
sample_shap = load_shap_attributions(binary_shap_path, ['S001'], n_classes=2)
print("Keys:", list(sample_shap['S001'].keys()))
print("Kappa keys:", list(sample_shap['S001']['kappa'].keys()))
print("mean_by_class keys:", list(sample_shap['S001']['kappa']['mean_by_class'].keys()))
first_key = list(sample_shap['S001']['kappa']['mean_by_class'].keys())[0]
print(f"Shape of class {first_key}:", np.array(sample_shap['S001']['kappa']['mean_by_class'][first_key]).shape)

Loaded: 1/1 subjects
Keys: ['kappa', 'composite']
Kappa keys: ['mean_by_class', 'n_trials']
mean_by_class keys: [0, 1]
Shape of class 0: (16, 480)


In [103]:
sample_mc_shap = load_shap_attributions(
    multiclass_shap_path, ['S018'], n_classes=4)
for cls in range(4):
    arr = np.array(sample_mc_shap['S018']['kappa']['mean_by_class'][cls])
    print(f"S018 class {cls}: shape={arr.shape}, ndim={arr.ndim}")

Loaded: 1/1 subjects
S018 class 0: shape=(16, 480), ndim=2
S018 class 1: shape=(16, 480), ndim=2
S018 class 2: shape=(), ndim=0
S018 class 3: shape=(16, 480), ndim=2


##### Shap Binary

In [97]:
# Load SHAP binary all 106 subjects
binary_shap_106 = load_shap_attributions(
    binary_shap_path, valid_subjects, n_classes=2)
print(f"Loaded: {len(binary_shap_106)} subjects")

# Build per-subject channel averages
shap_channel_avg = {}
for subj in valid_subjects:
    shap_channel_avg[subj] = {
        'class_0': np.mean(binary_shap_106[subj]['kappa']['mean_by_class'][0], axis=1),
        'class_1': np.mean(binary_shap_106[subj]['kappa']['mean_by_class'][1], axis=1)
    }

# Build relevance score table
rows_shap = []
for group in ['high', 'mid_high', 'mid_low', 'low', 'negative_transfer']:
    group_subjects = [s for s, g in score_groups.items() if g == group]
    for cls, cls_name in [(0, 'left'), (1, 'right')]:
        group_attrs = np.array([shap_channel_avg[s][f'class_{cls}']
                                for s in group_subjects])
        group_mean = np.mean(group_attrs, axis=0)
        for ch_idx, ch_name in enumerate(ch16_names):
            rows_shap.append({
                'group': group,
                'class': cls_name,
                'channel': ch_name,
                'mean_relevance': round(group_mean[ch_idx], 6)
            })

relevance_df_shap = pd.DataFrame(rows_shap)

for cls_name in ['left', 'right']:
    print(f"\nSHAP - Class: {cls_name}")
    pivot = relevance_df_shap[relevance_df_shap['class'] == cls_name].pivot(
        index='channel', columns='group', values='mean_relevance'
    )[['high', 'mid_high', 'mid_low', 'low', 'negative_transfer']]
    pivot = pivot.reindex(ch16_names)
    pivot.columns = ['High', 'Mid-High', 'Mid-Low', 'Low', 'Neg. Transfer']
    print(pivot.to_string())

Loaded: 106/106 subjects
Loaded: 106 subjects

SHAP - Class: left
             High  Mid-High   Mid-Low       Low  Neg. Transfer
channel                                                       
C3       0.000395  0.000110  0.000050 -0.000118      -0.000332
Cz      -0.000002  0.000002 -0.000019  0.000063       0.000048
C4       0.000530  0.000658  0.000530  0.000310       0.000341
FC3     -0.000042 -0.000021 -0.000060 -0.000058      -0.000069
FCz      0.000049 -0.000010  0.000046  0.000104       0.000051
CP3      0.000307  0.000050  0.000096 -0.000099      -0.000190
CPz     -0.000067 -0.000051  0.000018 -0.000008      -0.000031
CP4      0.000642  0.000586  0.000578  0.000411       0.000552
CP5      0.000011 -0.000004  0.000002 -0.000017      -0.000179
CP6      0.000078  0.000095  0.000039  0.000091       0.000264
P3       0.000078 -0.000023 -0.000005 -0.000019      -0.000005
P4       0.000108  0.000063  0.000048  0.000143       0.000012
C5      -0.000066 -0.000076 -0.000009  0.000056     

##### Multiclass - chap + IG commented

In [104]:
# Load SHAP and IG multiclass
mc_shap_106 = load_shap_attributions(
    multiclass_shap_path, valid_subjects, n_classes=4)
# mc_ig_106 = load_ig_attributions(
#     multiclass_ig_zero_path, valid_subjects, n_classes=4)
# print(f"SHAP loaded: {len(mc_shap_106)}, IG loaded: {len(mc_ig_106)}")

def build_mc_relevance_table(attrs_106, method_name):
    # Build per-subject channel averages with zeros for empty arrays
    channel_avg = {}
    for subj in valid_subjects:
        channel_avg[subj] = {}
        for cls in range(4):
            arr = attrs_106[subj]['kappa']['mean_by_class'][cls]
            if np.array(arr).ndim == 2:
                channel_avg[subj][cls] = np.mean(arr, axis=1)
            else:
                channel_avg[subj][cls] = np.zeros(16)
                print(f"Warning: {subj} class {cls} empty, replaced with zeros")

    rows = []
    for group in ['neither', 'group_a_collapse', 'group_b_flat', 'negative_transfer']:
        group_subjects = [s for s, g in category_groups_106.items() if g == group]
        for cls in range(4):
            group_attrs = np.array([channel_avg[s][cls] for s in group_subjects])
            group_mean = np.mean(group_attrs, axis=0)
            for ch_idx, ch_name in enumerate(ch16_names):
                rows.append({
                    'group': group,
                    'class': class_names_mc[cls],
                    'channel': ch_name,
                    'mean_relevance': round(group_mean[ch_idx], 6)
                })

    df = pd.DataFrame(rows)
    for cls_name in ['left', 'right', 'both_fists', 'both_feet']:
        print(f"\n{method_name} Multiclass - Class: {cls_name}")
        pivot = df[df['class'] == cls_name].pivot(
            index='channel', columns='group', values='mean_relevance'
        )[['neither', 'group_a_collapse', 'group_b_flat', 'negative_transfer']]
        pivot = pivot.reindex(ch16_names)
        pivot.columns = ['Neither', 'Group A', 'Group B', 'Neg. Transfer']
        print(pivot.to_string())
    return df

print("\n" + "="*60)
print("SHAP MULTICLASS")
print("="*60)
relevance_df_mc_shap = build_mc_relevance_table(mc_shap_106, "SHAP")

# print("\n" + "="*60)
# print("IG MULTICLASS")
# print("="*60)
# relevance_df_mc_ig = build_mc_relevance_table(mc_ig_106, "IG")

Loaded: 106/106 subjects

SHAP MULTICLASS

SHAP Multiclass - Class: left
          Neither   Group A       Group B  Neg. Transfer
channel                                                 
C3       0.000491  0.000049 -6.890000e-04      -0.000326
Cz       0.000007  0.000077 -8.800000e-05       0.000217
C4       0.000585  0.000523  1.055000e-03       0.001019
FC3      0.000005 -0.000134  2.050000e-04      -0.000273
FCz      0.000065  0.000039  1.770000e-04       0.000041
CP3      0.000481  0.000462 -4.380000e-04      -0.000622
CPz      0.000022 -0.000104 -1.700000e-05       0.000086
CP4      0.000488  0.000510  6.110000e-04       0.000656
CP5     -0.000038 -0.000030  1.590000e-04      -0.000031
CP6      0.000060  0.000147  1.580000e-04       0.000020
P3       0.000072 -0.000025 -3.800000e-05       0.000038
P4       0.000028  0.000029  9.000000e-05      -0.000012
C5      -0.000048 -0.000121  5.000000e-06      -0.000092
C6      -0.000002  0.000039  2.390000e-04       0.000160
P5      -0.0000

#### IG Relevance

##### Debug and checks

In [98]:
sample_ig = load_ig_attributions(binary_ig_zero_path, ['S001'], n_classes=2)
print("Keys:", list(sample_ig['S001'].keys()))
print("Kappa keys:", list(sample_ig['S001']['kappa'].keys()))
print("mean_by_class keys:", list(sample_ig['S001']['kappa']['mean_by_class'].keys()))
first_key = list(sample_ig['S001']['kappa']['mean_by_class'].keys())[0]
print(f"Shape of class {first_key}:", np.array(sample_ig['S001']['kappa']['mean_by_class'][first_key]).shape)

Loaded: 1/1 subjects
Keys: ['kappa', 'composite']
Kappa keys: ['mean_by_class', 'n_trials']
mean_by_class keys: [0, 1]
Shape of class 0: (16, 480)


##### IG Binary

In [99]:
# Load IG binary all 106 subjects
binary_ig_106 = load_ig_attributions(
    binary_ig_zero_path, valid_subjects, n_classes=2)
print(f"Loaded: {len(binary_ig_106)} subjects")

# Build per-subject channel averages
ig_channel_avg = {}
for subj in valid_subjects:
    ig_channel_avg[subj] = {
        'class_0': np.mean(binary_ig_106[subj]['kappa']['mean_by_class'][0], axis=1),
        'class_1': np.mean(binary_ig_106[subj]['kappa']['mean_by_class'][1], axis=1)
    }

# Build relevance score table
rows_ig = []
for group in ['high', 'mid_high', 'mid_low', 'low', 'negative_transfer']:
    group_subjects = [s for s, g in score_groups.items() if g == group]
    for cls, cls_name in [(0, 'left'), (1, 'right')]:
        group_attrs = np.array([ig_channel_avg[s][f'class_{cls}']
                                for s in group_subjects])
        group_mean = np.mean(group_attrs, axis=0)
        for ch_idx, ch_name in enumerate(ch16_names):
            rows_ig.append({
                'group': group,
                'class': cls_name,
                'channel': ch_name,
                'mean_relevance': round(group_mean[ch_idx], 6)
            })

relevance_df_ig = pd.DataFrame(rows_ig)

for cls_name in ['left', 'right']:
    print(f"\nIG - Class: {cls_name}")
    pivot = relevance_df_ig[relevance_df_ig['class'] == cls_name].pivot(
        index='channel', columns='group', values='mean_relevance'
    )[['high', 'mid_high', 'mid_low', 'low', 'negative_transfer']]
    pivot = pivot.reindex(ch16_names)
    pivot.columns = ['High', 'Mid-High', 'Mid-Low', 'Low', 'Neg. Transfer']
    print(pivot.to_string())

Loaded: 106/106 subjects
Loaded: 106 subjects

IG - Class: left
             High  Mid-High   Mid-Low       Low  Neg. Transfer
channel                                                       
C3       0.002303  0.002065  0.002066  0.001728       0.001000
Cz      -0.000003 -0.000013 -0.000046  0.000157      -0.000120
C4      -0.001498 -0.001367 -0.001493 -0.001517      -0.001072
FC3      0.000327  0.000300  0.000259  0.000343       0.000314
FCz     -0.000033 -0.000079 -0.000044  0.000025      -0.000100
CP3      0.002689  0.002154  0.002315  0.001809       0.001233
CPz      0.000020  0.000003  0.000131  0.000069       0.000199
CP4     -0.001462 -0.001405 -0.001489 -0.001634      -0.000957
CP5      0.000237  0.000248  0.000190  0.000288       0.000157
CP6     -0.000348 -0.000275 -0.000270 -0.000362      -0.000242
P3       0.000576  0.000374  0.000393  0.000366       0.000183
P4      -0.000374 -0.000261 -0.000311 -0.000200      -0.000177
C5      -0.000148 -0.000140 -0.000140 -0.000034       

##### Multiclass

In [105]:
mc_ig_106 = load_ig_attributions(
    multiclass_ig_zero_path, valid_subjects, n_classes=4)
print(f"SHAP loaded: {len(mc_shap_106)}, IG loaded: {len(mc_ig_106)}")

Loaded: 106/106 subjects
SHAP loaded: 106, IG loaded: 106


In [106]:
print("\n" + "="*60)
print("IG MULTICLASS")
print("="*60)
relevance_df_mc_ig = build_mc_relevance_table(mc_ig_106, "IG")


IG MULTICLASS

IG Multiclass - Class: left
          Neither   Group A   Group B  Neg. Transfer
channel                                             
C3       0.003213  0.002727  0.002819       0.002655
Cz       0.000056  0.000300  0.000031       0.000282
C4      -0.001925 -0.001932 -0.002613      -0.001837
FC3      0.000010 -0.000159 -0.000156      -0.000019
FCz      0.000008  0.000080  0.000016      -0.000021
CP3      0.002844  0.002283  0.003249       0.001816
CPz      0.000190  0.000068  0.000072       0.000172
CP4     -0.001514 -0.001486 -0.001394      -0.001546
CP5      0.000140  0.000061  0.000502       0.000162
CP6     -0.000263 -0.000197 -0.000142      -0.000387
P3       0.000444  0.000436  0.001068       0.000401
P4      -0.000267 -0.000298 -0.000204      -0.000318
C5      -0.000311 -0.000160 -0.000470      -0.000207
C6      -0.000100 -0.000166 -0.000145      -0.000090
P5      -0.000065  0.000021 -0.000334      -0.000044
P6      -0.000198 -0.000318 -0.000420      -0.000483

I

### misclassified trials checks 

In [ ]:
def run_misclassified_lrp_single(
    subject,
    checkpoint_tag="cdan_16ch_binary",
    stopping_criterion="kappa",
    n_classes=2,
    use_multiclass=False,
    use_16ch=True
):
    # Get LRP attributions
    correct, misclassified = compute_lrp(
    subject, checkpoint_tag, n_classes=n_classes,
    use_multiclass=use_multiclass,
    stopping_criterion=stopping_criterion,
    trial_type="both"
)

    # Unpack correct
    correct_per_trial = correct[0]
    correct_labels = correct[1]
    correct_mean = correct[2]

    # Unpack misclassified
    has_misclassified = len(misclassified[0]) > 0
    if has_misclassified:
        misc_per_trial = misclassified[0]
        misc_true_labels = misclassified[1]
        misc_pred_labels = misclassified[2]
        misc_mean = misclassified[3]
        misc_label_pairs = np.stack(
            [misc_true_labels, misc_pred_labels], axis=1)

    # Get confidence scores via forward pass
    model = load_model_for_xai(
        subject, checkpoint_tag,
        n_classes=n_classes,
        stopping_criterion=stopping_criterion
    )
    model.eval()

    _, _, target_eval_loader, _ = prepare_dataloaders(
        subject, valid_subjects,
        use_16ch=use_16ch,
        use_multiclass=use_multiclass
    )

    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for x, y in target_eval_loader:
            class_out, _, _ = model(x, alpha=0.0)
            probs = torch.softmax(class_out, dim=1)
            preds = torch.argmax(probs, dim=1)
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.numpy())

    all_probs = np.array(all_probs)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    if has_misclassified:
        misc_mask = all_preds != all_labels
        misc_confidence = all_probs[misc_mask]

    # Build save dict
    save_dict = {
        'correct_mean_class_0': correct_mean[0],
        'correct_mean_class_1': correct_mean[1],
        'correct_per_trial_attr': correct_per_trial,
        'correct_labels': correct_labels,
        'n_correct': len(correct_labels),
        'stopping_criterion': stopping_criterion,
        'n_classes': n_classes,
    }

    # Add remaining correct class means for multiclass
    for cls in range(2, n_classes):
        save_dict[f'correct_mean_class_{cls}'] = correct_mean.get(cls, np.zeros((16, 480)))

    if has_misclassified:
        save_dict.update({
            'misclassified_mean_class_0': misc_mean[0],
            'misclassified_mean_class_1': misc_mean[1],
            'misclassified_per_trial_attr': misc_per_trial,
            'misclassified_labels': misc_true_labels,
            'misclassified_label_pairs': misc_label_pairs,
            'misclassified_confidence': misc_confidence,
            'n_misclassified': len(misc_true_labels),
        })
        # Add remaining misclassified class means for multiclass
        for cls in range(2, n_classes):
            save_dict[f'misclassified_mean_class_{cls}'] = misc_mean.get(
                cls, np.zeros((16, 480)))
    else:
        save_dict.update({
            'misclassified_mean_class_0': np.zeros((16, 480)),
            'misclassified_mean_class_1': np.zeros((16, 480)),
            'misclassified_per_trial_attr': np.zeros((0, 16, 480)),
            'misclassified_labels': np.array([]),
            'misclassified_label_pairs': np.zeros((0, 2)),
            'misclassified_confidence': np.zeros((0, n_classes)),
            'n_misclassified': 0,
        })
        for cls in range(2, n_classes):
            save_dict[f'misclassified_mean_class_{cls}'] = np.zeros((16, 480))

    return save_dict


In [ ]:
# Save paths
binary_misc_path = results_path / "misclassified_attributions" / "binary_cdan_16ch"
multiclass_misc_path = results_path / "misclassified_attributions" / "multiclass_16ch"
binary_misc_path.mkdir(parents=True, exist_ok=True)
multiclass_misc_path.mkdir(parents=True, exist_ok=True)

def run_and_save_misclassified_lrp(
    subjects,
    save_path,
    checkpoint_tag="cdan_16ch_binary",
    stopping_criterion="kappa",
    n_classes=2,
    use_multiclass=False,
    use_16ch=True
):
    results_summary = {}
    start_total = time.time()

    for idx, subject in enumerate(subjects):
        start = time.time()
        try:
            save_dict = run_misclassified_lrp_single(
                subject,
                checkpoint_tag=checkpoint_tag,
                stopping_criterion=stopping_criterion,
                n_classes=n_classes,
                use_multiclass=use_multiclass,
                use_16ch=use_16ch
            )
            # Save
            np.savez(save_path / f"{subject}_misc_lrp.npz", **save_dict)
            elapsed = time.time() - start
            results_summary[subject] = {
                'n_correct': int(save_dict['n_correct']),
                'n_misclassified': int(save_dict['n_misclassified']),
                'time': round(elapsed, 2)
            }
            print(f"[{idx+1:3d}/106] {subject}: "
                  f"correct={save_dict['n_correct']}, "
                  f"misc={save_dict['n_misclassified']}, "
                  f"time={elapsed:.1f}s")
        except Exception as e:
            print(f"[{idx+1:3d}/106] {subject}: FAILED - {e}")
            results_summary[subject] = {'error': str(e)}

    total = time.time() - start_total
    print(f"\nDone. Total time: {total:.1f}s ({total/60:.1f} min)")
    print(f"Saved to: {save_path}")
    return results_summary



#### binary

In [ ]:
# Run binary 106 subjects
print("Running binary misclassified LRP for all 106 subjects...")
binary_summary = run_and_save_misclassified_lrp(
    valid_subjects,
    save_path=binary_misc_path,
    checkpoint_tag="cdan_16ch_binary",
    n_classes=2,
    use_multiclass=False,
    use_16ch=True
)

In [85]:
# Verify one binary saved file
sample_binary = np.load(binary_misc_path / "S006_misc_lrp.npz", allow_pickle=True)
print("Keys:", list(sample_binary.keys()))
print(f"n_correct: {sample_binary['n_correct']}")
print(f"n_misclassified: {sample_binary['n_misclassified']}")
print(f"n_correct + n_misclassified: {int(sample_binary['n_correct']) + int(sample_binary['n_misclassified'])}")
print(f"misclassified_confidence shape: {sample_binary['misclassified_confidence'].shape}")
print(f"misclassified_label_pairs shape: {sample_binary['misclassified_label_pairs'].shape}")

Keys: ['correct_mean_class_0', 'correct_mean_class_1', 'correct_per_trial_attr', 'correct_labels', 'n_correct', 'stopping_criterion', 'n_classes', 'misclassified_mean_class_0', 'misclassified_mean_class_1', 'misclassified_per_trial_attr', 'misclassified_labels', 'misclassified_label_pairs', 'misclassified_confidence', 'n_misclassified']
n_correct: 25
n_misclassified: 20
n_correct + n_misclassified: 45
misclassified_confidence shape: (20, 2)
misclassified_label_pairs shape: (20, 2)


#### Multiclass

##### Debug

In [ ]:
import inspect
from explainability import compute_lrp
# Print just the data loading part of compute_lrp
source = inspect.getsource(compute_lrp)
print(source[:3000])

def compute_lrp(
    subject,
    checkpoint_tag,
    n_classes=2,
    use_multiclass=False,
    stopping_criterion="kappa",
    first_layer_rule="epsilon",
    epsilon=1e-6,
    trial_type="correct",
):
    """
    Manual LRP implementation for DANNClassifierWrapper + EEGNet.
    trial_type:
        'correct'      — only correctly classified trials, attribution
                         targets true label (original behaviour)
        'misclassified'— only misclassified trials, attribution targets
                         predicted label (what model actually decided)
        'all'          — both sets returned separately as a tuple
    Returns:
        if trial_type != 'all':
            attributions: (n_trials, n_channels, n_timepoints)
            labels: (n_trials,) true labels
            mean_attr_per_class: dict {class_idx: mean attr (n_channels, n_timepoints)}
            [if misclassified: also stores predicted labels separately]
        if trial_type == 'all':
            (corr

In [ ]:
# Check how we are calling compute_lrp in run_misclassified_lrp_single
import inspect
print(inspect.getsource(run_misclassified_lrp_single))

def run_misclassified_lrp_single(
    subject,
    checkpoint_tag="cdan_16ch_binary",
    stopping_criterion="kappa",
    n_classes=2,
    use_multiclass=False,
    use_16ch=True
):
    # Get LRP attributions
    correct, misclassified = compute_lrp(
        subject, checkpoint_tag, n_classes=n_classes,
        stopping_criterion=stopping_criterion,
        trial_type="both"
    )

    # Unpack correct
    correct_per_trial = correct[0]
    correct_labels = correct[1]
    correct_mean = correct[2]

    # Unpack misclassified
    has_misclassified = len(misclassified[0]) > 0
    if has_misclassified:
        misc_per_trial = misclassified[0]
        misc_true_labels = misclassified[1]
        misc_pred_labels = misclassified[2]
        misc_mean = misclassified[3]
        misc_label_pairs = np.stack(
            [misc_true_labels, misc_pred_labels], axis=1)

    # Get confidence scores via forward pass
    model = load_model_for_xai(
        subject, checkpoint_tag,
        n_classes=n_

In [ ]:
# Debug S006 multiclass
_, _, target_eval_loader, _ = prepare_dataloaders(
    'S006', valid_subjects,
    use_16ch=True,
    use_multiclass=True
)

total_trials = 0
for x, y in target_eval_loader:
    total_trials += len(y)
print(f"Trials in eval loader: {total_trials}")
print(f"n_correct + n_misclassified from LRP: {15 + 30}")

Trials in eval loader: 90
n_correct + n_misclassified from LRP: 45


In [ ]:
multiclass_data = np.load(base_path / "Data/processed_multiclass_16ch/S006_multiclass_16ch.npz")
print("Keys:", list(multiclass_data.keys()))
print("Data shape:", multiclass_data['data'].shape)
print("Labels:", np.unique(multiclass_data['labels'], return_counts=True))

Keys: ['data', 'labels']
Data shape: (90, 16, 480)
Labels: (array([0, 1, 2, 3]), array([24, 21, 22, 23]))


##### verification

In [84]:
print("Testing multiclass on S006...")
result_mc = run_misclassified_lrp_single(
    'S006',
    checkpoint_tag="multiclass_16ch",
    n_classes=4,
    use_multiclass=True
)
print("Keys:", list(result_mc.keys()))
print(f"n_correct: {result_mc['n_correct']}")
print(f"n_misclassified: {result_mc['n_misclassified']}")
print(f"misclassified_label_pairs shape: {result_mc['misclassified_label_pairs'].shape}")
print(f"misclassified_confidence shape: {result_mc['misclassified_confidence'].shape}")

Testing multiclass on S006...
Keys: ['correct_mean_class_0', 'correct_mean_class_1', 'correct_per_trial_attr', 'correct_labels', 'n_correct', 'stopping_criterion', 'n_classes', 'correct_mean_class_2', 'correct_mean_class_3', 'misclassified_mean_class_0', 'misclassified_mean_class_1', 'misclassified_per_trial_attr', 'misclassified_labels', 'misclassified_label_pairs', 'misclassified_confidence', 'n_misclassified', 'misclassified_mean_class_2', 'misclassified_mean_class_3']
n_correct: 32
n_misclassified: 58
misclassified_label_pairs shape: (58, 2)
misclassified_confidence shape: (58, 4)


##### after verification

In [ ]:
print("Running multiclass misclassified LRP for all 106 subjects...")
multiclass_summary = run_and_save_misclassified_lrp(
    valid_subjects,
    save_path=multiclass_misc_path,
    checkpoint_tag="multiclass_16ch",
    n_classes=4,
    use_multiclass=True,
    use_16ch=True
)

In [87]:
sample_mc = np.load(multiclass_misc_path / "S006_misc_lrp.npz", allow_pickle=True)
print("Keys:", list(sample_mc.keys()))
print(f"n_correct: {sample_mc['n_correct']}")
print(f"n_misclassified: {sample_mc['n_misclassified']}")
print(f"n_correct + n_misclassified: {int(sample_mc['n_correct']) + int(sample_mc['n_misclassified'])}")
print(f"misclassified_confidence shape: {sample_mc['misclassified_confidence'].shape}")
print(f"misclassified_label_pairs shape: {sample_mc['misclassified_label_pairs'].shape}")
print(f"n_classes: {sample_mc['n_classes']}")

Keys: ['correct_mean_class_0', 'correct_mean_class_1', 'correct_per_trial_attr', 'correct_labels', 'n_correct', 'stopping_criterion', 'n_classes', 'correct_mean_class_2', 'correct_mean_class_3', 'misclassified_mean_class_0', 'misclassified_mean_class_1', 'misclassified_per_trial_attr', 'misclassified_labels', 'misclassified_label_pairs', 'misclassified_confidence', 'n_misclassified', 'misclassified_mean_class_2', 'misclassified_mean_class_3']
n_correct: 32
n_misclassified: 58
n_correct + n_misclassified: 90
misclassified_confidence shape: (58, 4)
misclassified_label_pairs shape: (58, 2)
n_classes: 4


In [91]:
# Test refactored function with explicit valid_subjects parameter
print("Testing refactored function on S006 binary...")
result_test = run_misclassified_lrp_single(
    'S006',
    valid_subjects,
    checkpoint_tag="cdan_16ch_binary",
    n_classes=2,
    use_multiclass=False
)
print(f"n_correct: {result_test['n_correct']}")
print(f"n_misclassified: {result_test['n_misclassified']}")
print(f"n_correct + n_misclassified: {result_test['n_correct'] + result_test['n_misclassified']}")

Testing refactored function on S006 binary...
n_correct: 25
n_misclassified: 20
n_correct + n_misclassified: 45


#### Debug

In [ ]:
# Check what is stored in one LRP attribution file
sample = np.load(binary_attr_path / "S007_lrp.npz")
print("Keys:", list(sample.keys()))
for key in sample.keys():
    arr = np.array(sample[key])
    print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")

Keys: ['n_classes', 'labels_kappa', 'n_trials_kappa', 'mean_attr_class_0_kappa', 'mean_attr_class_1_kappa', 'labels_composite', 'n_trials_composite', 'mean_attr_class_0_composite', 'mean_attr_class_1_composite']
  n_classes: shape=(), dtype=int64
  labels_kappa: shape=(45,), dtype=int64
  n_trials_kappa: shape=(), dtype=int64
  mean_attr_class_0_kappa: shape=(16, 480), dtype=float32
  mean_attr_class_1_kappa: shape=(16, 480), dtype=float32
  labels_composite: shape=(45,), dtype=int64
  n_trials_composite: shape=(), dtype=int64
  mean_attr_class_0_composite: shape=(16, 480), dtype=float32
  mean_attr_class_1_composite: shape=(16, 480), dtype=float32


In [ ]:
import time

test_subject = 'S007'

start = time.time()

# compute_lrp handles model loading and data internally
attr_result = compute_lrp(
    test_subject,
    "cdan_16ch_binary",
    n_classes=2,
    trial_type="both"  # get both correct and misclassified
)

elapsed = time.time() - start
print(f"Time for one subject: {elapsed:.1f} seconds")
print(f"Estimated for 13 subjects: {elapsed * 13 / 60:.1f} minutes")
print(f"Estimated for 106 subjects: {elapsed * 106 / 60:.1f} minutes")

Time for one subject: 0.3 seconds
Estimated for 13 subjects: 0.1 minutes
Estimated for 106 subjects: 0.5 minutes


In [ ]:
print("Type:", type(attr_result))
if isinstance(attr_result, tuple):
    print("Tuple length:", len(attr_result))
    for i, item in enumerate(attr_result):
        print(f"  Item {i}: type={type(item)}")
        if isinstance(item, tuple):
            for j, subitem in enumerate(item):
                print(f"    Subitem {j}: type={type(subitem)}, ", end="")
                if hasattr(subitem, 'shape'):
                    print(f"shape={subitem.shape}")
                else:
                    print(f"value={subitem}")

Type: <class 'tuple'>
Tuple length: 2
  Item 0: type=<class 'tuple'>
    Subitem 0: type=<class 'numpy.ndarray'>, shape=(45, 16, 480)
    Subitem 1: type=<class 'numpy.ndarray'>, shape=(45,)
    Subitem 2: type=<class 'dict'>, value={0: array([[-1.01640398e-05, -1.38903284e-04, -2.54540821e-04, ...,
         3.57097900e-03,  3.02226236e-03,  2.18025572e-03],
       [-3.28448688e-04, -9.33775082e-05, -1.23283389e-04, ...,
        -2.50369107e-04, -2.50905228e-04, -4.50855412e-04],
       [ 2.30348519e-06,  8.50761353e-05, -2.11383958e-04, ...,
         1.59738251e-04, -1.01158534e-04, -2.23355251e-04],
       ...,
       [-2.57332256e-04, -4.30235523e-04, -2.41867179e-04, ...,
        -2.15191554e-04, -4.78553484e-05, -9.86526720e-05],
       [-1.44235819e-04, -1.95918910e-04, -2.02366311e-04, ...,
        -1.40694814e-04, -7.12857291e-04, -5.53669815e-04],
       [-1.25577557e-04, -3.29892355e-04, -6.88566302e-04, ...,
        -7.35561407e-05,  2.04274125e-04,  1.83077325e-04]],
      

In [ ]:
# Test on a lower performing subject
test_subject = 'S006'  # Group A collapse

start = time.time()
attr_result_s006 = compute_lrp(
    test_subject,
    "cdan_16ch_binary",
    n_classes=2,
    trial_type="both"
)
elapsed = time.time() - start

correct, misclassified = attr_result_s006
print(f"Time: {elapsed:.1f} seconds")
print(f"Correct trials: {correct[0].shape[0] if len(correct[0].shape) > 0 else 0}")
print(f"Misclassified trials: {misclassified[0].shape[0]}")
print(f"Misclassified class 0 attribution: {misclassified[3][0]}")
print(f"Misclassified class 1 attribution: {misclassified[3][1]}")

Time: 0.3 seconds
Correct trials: 25
Misclassified trials: 20
Misclassified class 0 attribution: [[ 1.33223215e-03  1.60362315e-03  2.56015512e-04 ...  7.42782868e-05
  -1.02076374e-04  2.94124649e-04]
 [-8.87511778e-05  2.46143554e-05  9.84614380e-05 ...  2.72285368e-04
   3.75491509e-04  3.91752328e-05]
 [-5.71181008e-04 -6.83208171e-04 -1.17826287e-03 ... -1.70670054e-03
  -7.12564855e-04 -8.90049152e-04]
 ...
 [ 5.40667563e-04  6.87816413e-04  1.04090164e-03 ... -4.19808232e-04
  -2.99273415e-05  5.32650520e-05]
 [-2.95572827e-04 -1.29919936e-04  5.19664376e-04 ... -3.18726612e-04
  -2.52637226e-04 -1.88670005e-04]
 [ 7.62524869e-05  3.81984340e-04  7.62010866e-04 ...  4.04864768e-05
   2.04165815e-04  1.90593739e-04]]
Misclassified class 1 attribution: [[ 4.00914258e-04  5.73320631e-06 -5.72557561e-04 ...  2.77247600e-04
  -2.02155679e-05  5.02666990e-05]
 [-1.34759015e-04 -1.52501059e-04 -6.05126770e-05 ... -1.22722895e-05
  -7.38962044e-05 -1.39503420e-04]
 [-9.80089069e-04  2.2

In [ ]:
correct, misclassified = attr_result_s006

print("CORRECT tuple:")
for i, item in enumerate(correct):
    if isinstance(item, np.ndarray):
        print(f"  Item {i}: shape={item.shape}, dtype={item.dtype}")
    elif isinstance(item, dict):
        print(f"  Item {i}: dict, keys={list(item.keys())}")
        for k, v in item.items():
            if v is not None:
                print(f"    {k}: shape={np.array(v).shape}")
            else:
                print(f"    {k}: None")
    else:
        print(f"  Item {i}: type={type(item)}, value={item}")

print("\nMISCLASSIFIED tuple:")
for i, item in enumerate(misclassified):
    if isinstance(item, np.ndarray):
        print(f"  Item {i}: shape={item.shape}, dtype={item.dtype}")
    elif isinstance(item, dict):
        print(f"  Item {i}: dict, keys={list(item.keys())}")
        for k, v in item.items():
            if v is not None:
                print(f"    {k}: shape={np.array(v).shape}")
            else:
                print(f"    {k}: None")
    else:
        print(f"  Item {i}: type={type(item)}, value={item}")

CORRECT tuple:
  Item 0: shape=(25, 16, 480), dtype=float32
  Item 1: shape=(25,), dtype=int64
  Item 2: dict, keys=[0, 1]
    0: shape=(16, 480)
    1: shape=(16, 480)

MISCLASSIFIED tuple:
  Item 0: shape=(20, 16, 480), dtype=float32
  Item 1: shape=(20,), dtype=int64
  Item 2: shape=(20,), dtype=int64
  Item 3: dict, keys=[0, 1]
    0: shape=(16, 480)
    1: shape=(16, 480)


In [ ]:
# Check what checkpoint folders exist
checkpoint_path = results_path / "checkpoints"
print("Checkpoint folders:")
for folder in sorted(checkpoint_path.iterdir()):
    print(f"  {folder.name}")

Checkpoint folders:
  .DS_Store
  .gitkeep
  cdan_16ch_binary
  multiclass_16ch


### Refactor missclassified to py 

In [112]:
import inspect
from explainability import compute_shap, compute_ig, compute_deeplift
print("compute_shap args:", inspect.signature(compute_shap))
print("compute_ig args:", inspect.signature(compute_ig))
print("compute_deeplift args:", inspect.signature(compute_deeplift))

compute_shap args: (subject, checkpoint_tag, valid_subjects, n_classes=2, use_multiclass=False, stopping_criterion='kappa', background_fraction=0.6, trials_per_source_subject=26, trial_type='correct', seed=25)
compute_ig args: (subject, checkpoint_tag, n_classes=2, use_multiclass=False, stopping_criterion='kappa', baseline='zero', n_steps=50, trial_type='correct')
compute_deeplift args: (subject, checkpoint_tag, n_classes=2, use_multiclass=False, stopping_criterion='kappa', baseline='mean', trial_type='correct')
